# After tracking: fix swaps, re-join players, read shirt numbers

Run this after `full_match.ipynb` has finished for the match. It:
1. **Cuts the tracker's mistakes** where two players of different teams crossed and the tracker swapped
   them, or a track drifted onto the other team (shirt colour before vs after; each team's kit is learnt
   from the video). A cut only splits a track; it never merges two players.
2. **Re-joins the pieces into players** with the fixed tracks (same method as `full_match.ipynb`).
3. **Reads shirt numbers** on every piece (best close views, enhanced, digit reader), to name players
   automatically, mostly the opponent.
4. **Refreshes the data for the Pass Review app.**

The original joining is kept as `match_tracks_unfixed.csv.gz`.

**Runtime:** L4 GPU, about 3-6 minutes per segment. **Runtime -> Run all**; do not edit cells.
If it disconnects, **Run all again**: finished segments are skipped.

In [ ]:
# Cell 1 - GPU, Drive, inputs
MATCH = 'ittifaq_faysali'  #@param {type:"string"}
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
OUT = f'{PBD}/full_match/{MATCH}'; SEG = f'{OUT}/segments'; REVIEW = f'{PBD}/review/{MATCH}'
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
NAMES = [os.path.basename(c)[:-4] for c in CLIPS]
missing = [n for n in NAMES if not os.path.exists(f'{SEG}/{n}/per_frame_tracks.csv.gz')]
if not CLIPS or missing or not os.path.exists(f'{OUT}/match_tracks.csv.gz'):
    raise SystemExit(f'Run full_match.ipynb for {MATCH} first (missing: {missing[:3] or "match_tracks.csv.gz"}).')
print(len(CLIPS), 'segments')

In [ ]:
# Cell 2 - Our repo and packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q easyocr "transformers>=4.47"
import numpy as np, pandas as pd, torch, cv2
from tools.crossings import find_crossings, sample_frames
from tools.crossing_fix import kit_signature, team_kits, kit_prob, crossing_cuts, flip_candidates, flip_cuts, apply_cuts
from tools.reid_appearance import Embedder, crops_from_video, sample_rows
emb_model = Embedder(); emb_model.load_state_dict(torch.load(MODEL, map_location=emb_model.device))
print('ready')

In [ ]:
# Cell 3 - Cut swaps and team switches, per segment (skips finished segments; ~1-2 min each)
log = {}
for clip, n in zip(CLIPS, NAMES):
    dst = f'{SEG}/{n}/per_frame_tracks_fixed.csv.gz'
    if os.path.exists(dst) and os.path.exists(f'{SEG}/{n}/pieces_emb_fixed.npz'):
        log[n] = json.load(open(f'{SEG}/{n}/fix_log.json')); continue
    t0 = time.time()
    t = pd.read_csv(f'{SEG}/{n}/per_frame_tracks.csv.gz', low_memory=False)
    people = t[t.class_id.isin((1, 2)) & (t.raw_tracker_id >= 0)]
    X = find_crossings(people)
    s = sample_frames(people, X).assign(src='cross')
    C = flip_candidates(people)
    fr = []
    for i, r in C.iterrows():
        g = people[people.raw_tracker_id == r.piece].set_index('frame')
        for when, fs in (('b', [r.frame - 30, r.frame - 15]), ('a', [r.frame + 15, r.frame + 30])):
            for f in fs:
                if f in g.index:
                    b = g.loc[f]; b = b.iloc[0] if isinstance(b, pd.DataFrame) else b
                    fr.append((i, when, int(f), b.x1, b.y1, b.x2, b.y2))
    fr = pd.DataFrame(fr, columns=['cand', 'when', 'frame', 'x1', 'y1', 'x2', 'y2']).assign(src='flip')
    known = people[people.team_id.isin((0, 1))]
    kit = known.groupby('team_id', group_keys=False).apply(lambda g: g.sample(min(len(g), 200), random_state=0)).assign(src='kit')
    rows = pd.concat([s, fr, kit[['frame', 'x1', 'y1', 'x2', 'y2', 'team_id', 'src']]], ignore_index=True)
    rows['row'] = np.arange(len(rows))
    crops, rows2 = crops_from_video(clip, rows)
    sig = np.zeros((len(rows), 16), np.float32)
    sig[rows2.row.to_numpy()] = np.stack([kit_signature(c) for c in crops]) if len(crops) else sig
    k = rows.src == 'kit'
    kits = team_kits(sig[k.to_numpy()], rows.team_id[k].to_numpy())
    prob = np.array([kit_prob(v, kits) for v in sig])
    sc = (rows.src == 'cross').to_numpy()
    cuts, dec = crossing_cuts(people, X, rows[sc].reset_index(drop=True), prob[sc])
    fl = rows.src == 'flip'
    bp = {i: list(prob[(fl & (rows.cand == i) & (rows.when == 'b')).to_numpy()]) for i in C.index}
    ap = {i: list(prob[(fl & (rows.cand == i) & (rows.when == 'a')).to_numpy()]) for i in C.index}
    for p, fs in flip_cuts(C, bp, ap).items():
        cuts.setdefault(p, set()).update(fs)
    fixed, new = apply_cuts(t, cuts)
    fixed.to_csv('/content/fixed.csv.gz', index=False, compression='gzip'); shutil.copy('/content/fixed.csv.gz', dst)
    # appearance of the pieces that changed (cut pieces and their new tails); the others keep theirs
    old = np.load(f'{SEG}/{n}/pieces_emb.npz')
    changed = set(cuts) | set(new)
    keep = ~np.isin(old['piece'], list(changed))
    fp = fixed[fixed.raw_tracker_id.isin(changed) & fixed.class_id.isin((1, 2))]
    if len(fp):
        cr, rr = crops_from_video(clip, sample_rows(fp, 'raw_tracker_id', per_id=12))
        e_new, p_new = emb_model.embed(cr).astype(np.float16), rr.raw_tracker_id.to_numpy()
    else:
        e_new, p_new = np.zeros((0, old['emb'].shape[1]), np.float16), np.zeros(0, int)
    np.savez_compressed('/content/emb.npz', emb=np.concatenate([old['emb'][keep], e_new]), piece=np.concatenate([old['piece'][keep], p_new]))
    shutil.copy('/content/emb.npz', f'{SEG}/{n}/pieces_emb_fixed.npz')
    log[n] = {'crossings': len(X), 'cross_team_checked': int((dec.decision != 'skipped').sum()) if len(dec) else 0,
              'swaps_cut': int((dec.decision == 'cut').sum()) if len(dec) else 0, 'team_switch_candidates': len(C),
              'pieces_cut': len(cuts), 'new_pieces': len(new), 'kits_learnt': sorted(int(x) for x in kits)}
    json.dump(log[n], open(f'{SEG}/{n}/fix_log.json', 'w'))
    print(n, log[n], f'{(time.time() - t0) / 60:.1f} min')
json.dump(log, open(f'{OUT}/fix_log.json', 'w'), indent=1)
print('fixes done:', sum(v['swaps_cut'] for v in log.values()), 'swaps cut,', sum(v['pieces_cut'] for v in log.values()), 'pieces cut')

In [ ]:
# Cell 4 - Re-join the pieces into players with the fixed tracks (same method as full_match.ipynb)
from tools.full_match import ID_STRIDE, merge_segments, team_mapping, apply_team_mapping
from tools.join_pieces import piece_table, cluster, joined_ids
FINAL = f'{OUT}/match_tracks.csv.gz'
if not os.path.exists(f'{OUT}/match_tracks_unfixed.csv.gz'):
    shutil.copy(FINAL, f'{OUT}/match_tracks_unfixed.csv.gz')
segs, embs, crop_piece = [], [], []
for i, name in enumerate(NAMES):
    t = pd.read_csv(f'{SEG}/{name}/per_frame_tracks_fixed.csv.gz', low_memory=False)
    nfr = json.load(open(f'{SEG}/{name}/kpi_summary.json'))['processed_frames']
    segs.append((name, t, nfr))
    d = np.load(f'{SEG}/{name}/pieces_emb_fixed.npz')
    embs.append(d['emb'].astype(np.float32)); crop_piece.append(d['piece'] + i * ID_STRIDE)
match = merge_segments(segs); del segs
emb, crop_piece = np.concatenate(embs), np.concatenate(crop_piece)
pieces = piece_table(match)
seg_of = match[match.raw_tracker_id >= 0].drop_duplicates('raw_tracker_id').set_index('raw_tracker_id').segment.to_dict()
maps = team_mapping(emb, crop_piece, {p: v['team'] for p, v in pieces.items()}, seg_of, NAMES)
match = apply_team_mapping(match, maps)
pieces = piece_table(match)
lab = cluster(pieces, emb, crop_piece, thr=0.3)
match['joined_id'] = joined_ids(match, lab)
people = match.class_id.isin((1, 2)) & (match.joined_id > 0)
match.loc[people, 'display_track_id'] = match.loc[people, 'joined_id']
match.to_csv('/content/match_fixed.csv.gz', index=False, compression='gzip'); shutil.copy('/content/match_fixed.csv.gz', FINAL)
before = pd.read_csv(f'{OUT}/match_tracks_unfixed.csv.gz', usecols=['class_id', 'joined_id'])
summary = {'players_joined_before': int(before.joined_id[before.class_id.isin((1, 2)) & (before.joined_id > 0)].nunique()),
           'players_joined_after': int(match.joined_id[people].nunique()), 'pieces': len(pieces)}
json.dump(summary, open(f'{OUT}/fix_summary.json', 'w'), indent=1)
print(summary)

In [ ]:
# Cell 5 - Shirt numbers: best close views of every piece, enhanced, read by a digit reader (~2-4 min per segment)
import easyocr
from tools.jersey import best_views, torso_crop, enhance, clean_number, vote
reader = easyocr.Reader(['en'], gpu=True, verbose=False)
match = pd.read_csv(FINAL, low_memory=False, usecols=['frame', 'segment', 'segment_frame', 'raw_tracker_id', 'joined_id', 'class_id', 'team_id', 'x1', 'y1', 'x2', 'y2'])
for clip, n in zip(CLIPS, NAMES):
    dst = f'{REVIEW}/{n}/jersey_reads_fixed.csv.gz'
    if os.path.exists(dst):
        continue
    t0 = time.time()
    t = match[(match.segment == n) & match.class_id.isin((1, 2)) & (match.raw_tracker_id >= 0)]
    views = []
    for piece, g in t.groupby('raw_tracker_id'):
        if len(g) >= 15:
            views.append(best_views(g.reset_index(drop=True), k=6).assign(piece=piece))
    V = pd.concat(views, ignore_index=True) if views else t.iloc[:0].assign(piece=[])
    want = {int(f): grp for f, grp in V.groupby('segment_frame')}
    cap = cv2.VideoCapture(clip); reads = []; f = 0
    for target in sorted(want):
        while f < target:
            cap.grab(); f += 1
        ok, frame = cap.read(); f += 1
        if not ok:
            break
        for r in want[target].itertuples():
            img = enhance(torso_crop(frame, (r.x1, r.y1, r.x2, r.y2)))
            res = reader.readtext(img, allowlist='0123456789', detail=1, paragraph=False, min_size=4, text_threshold=0.5)
            best = max(res, key=lambda x: x[2], default=None)
            reads.append((int(r.piece), int(r.joined_id), int(r.team_id), target, clean_number(best[1]) if best else None, float(best[2]) if best else 0.0))
    cap.release()
    R = pd.DataFrame(reads, columns=['piece', 'joined_id', 'team_id', 'segment_frame', 'number', 'conf'])
    os.makedirs(f'{REVIEW}/{n}', exist_ok=True)
    R.to_csv('/content/jr.csv.gz', index=False, compression='gzip'); shutil.copy('/content/jr.csv.gz', dst)
    print(n, len(R), 'reads,', R.number.notna().sum(), 'with a number', f'{(time.time() - t0) / 60:.1f} min')
allr = pd.concat([pd.read_csv(f'{REVIEW}/{n}/jersey_reads_fixed.csv.gz').assign(segment=n) for n in NAMES], ignore_index=True)
allr.to_csv(f'{REVIEW}/jersey_reads_all.csv', index=False)
votes = vote(allr.rename(columns={'piece': 'piece'}))
votes.to_csv(f'{REVIEW}/jersey_votes.csv', index=False)
print(len(votes), 'pieces with a voted number')

In [ ]:
# Cell 6 - Data for the Pass Review app (rewritten from the fixed tracks)
REVIEW_COLS = ['frame', 'segment', 'segment_frame', 'raw_tracker_id', 'joined_id', 'class_id', 'team_id', 'x1', 'y1', 'x2', 'y2', 'x_m', 'y_m']
t = pd.read_csv(FINAL, low_memory=False, usecols=lambda c: c in REVIEW_COLS)
for clip, n in zip(CLIPS, NAMES):
    g = t[t.segment == n].drop(columns='segment').copy()
    for c in ('x1', 'y1', 'x2', 'y2'):
        g[c] = g[c].round().astype(int)
    for c in ('x_m', 'y_m'):
        g[c] = g[c].round()
    os.makedirs(f'{REVIEW}/{n}', exist_ok=True)
    g.to_csv(f'{REVIEW}/{n}/tracks_small.csv.gz', index=False, compression='gzip')
    cap = cv2.VideoCapture(clip)
    json.dump({'segment': n, 'seg_start': int(g.frame.min() - g.segment_frame.min()),
               'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 'fps': cap.get(cv2.CAP_PROP_FPS),
               'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))},
              open(f'{REVIEW}/{n}/segment.json', 'w'))
    cap.release()
    shutil.copy(f'{SEG}/{n}/pieces_emb_fixed.npz', f'{REVIEW}/{n}/pieces_emb.npz')
print(f'Done. Tell Claude: after-tracking finished for {MATCH}.')